# Bye-Bye Lanternfly — Model Evaluation

Evaluate a trained YOLO model on the held-out test split and calculate image-level count metrics. Run this after training, or point `WEIGHTS` to an existing checkpoint.

**Important:** test results are only a reliable final estimate if the test split has not been used to choose checkpoints, thresholds, or settings. If you have already compared models on it, describe these results as exploratory.


In [ ]:
!pip -q install ultralytics==8.4.202 pyyaml pandas


In [ ]:
from pathlib import Path
import yaml
import pandas as pd
from ultralytics import YOLO

from google.colab import drive
drive.mount("/content/drive")

# Update these paths for your Drive setup.
DATASET_DIR = Path("/content/lanternfly_v4_fixed")
WEIGHTS = Path("/content/drive/MyDrive/ByeByeLanternfly/models/best.pt")
OUTPUT_DIR = Path("/content/drive/MyDrive/ByeByeLanternfly/evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_YAML = DATASET_DIR / "data.yaml"
assert DATA_YAML.exists(), f"Missing dataset YAML: {DATA_YAML}"
assert WEIGHTS.exists(), f"Missing model checkpoint: {WEIGHTS}"

with DATA_YAML.open("r", encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

test_images = Path(data_config["path"]) / data_config.get("test", "test/images")
print("Test images:", test_images)


## Run standard detection validation

The confidence threshold below is configurable. If selecting a threshold, use validation data—not test data.


In [ ]:
model = YOLO(str(WEIGHTS))
metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    conf=0.20,
    iou=0.45,
    plots=True,
    project=str(OUTPUT_DIR),
    name="test_metrics",
    exist_ok=True
)

print("mAP50:", float(metrics.box.map50))
print("mAP50-95:", float(metrics.box.map))
print("Precision:", float(metrics.box.mp))
print("Recall:", float(metrics.box.mr))


## Calculate count metrics per image

This compares the number of predicted boxes with the number of annotated boxes in each test image. It is an image-count evaluation, not a population estimate.


In [ ]:
from collections import defaultdict
import math

CONFIDENCE = 0.20
IOU = 0.45
predictions = model.predict(
    source=str(test_images),
    imgsz=640,
    conf=CONFIDENCE,
    iou=IOU,
    max_det=1000,
    stream=True,
    verbose=False
)

rows = []
for result in predictions:
    image_path = Path(result.path)
    label_path = Path(str(image_path).replace("/images/", "/labels/")).with_suffix(".txt")
    if not label_path.exists():
        continue

    # YOLO label format: class_id x_center y_center width height
    ground_truth_count = sum(
        1 for line in label_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    )
    predicted_count = 0 if result.boxes is None else len(result.boxes)
    rows.append({
        "image": image_path.name,
        "actual_count": ground_truth_count,
        "predicted_count": predicted_count,
        "error": predicted_count - ground_truth_count,
        "absolute_error": abs(predicted_count - ground_truth_count)
    })

counts = pd.DataFrame(rows)
assert not counts.empty, "No labeled test images were evaluated. Check the dataset paths."

mae = counts["absolute_error"].mean()
rmse = math.sqrt((counts["error"] ** 2).mean())
exact_accuracy = (counts["error"] == 0).mean()
within_one = (counts["absolute_error"] <= 1).mean()

summary = pd.DataFrame([{
    "images_evaluated": len(counts),
    "MAE": mae,
    "RMSE": rmse,
    "exact_count_accuracy": exact_accuracy,
    "within_one_accuracy": within_one,
    "confidence": CONFIDENCE,
    "IoU": IOU
}])

display(summary)
display(counts.head())


In [ ]:
counts.to_csv(OUTPUT_DIR / "test_counts.csv", index=False)
summary.to_csv(OUTPUT_DIR / "evaluation_summary.csv", index=False)
print("Saved reports to:", OUTPUT_DIR)
